#### GOLD LAYER — DimProducts (SCD Type 2 via Delta Live Tables)
###### Declarative pipeline: silver → staging → view → SCD2 dimension
###### apply_changes replaces the entire manual upsert.

In [0]:
import dlt

In [0]:
#Data quality expectations (the "contract" for this table)
#    Each entry: rule_name -> SQL condition every row must pass.
my_rules = {
    "valid_product_id": "product_id is not null",
    "valid_product_name": "product_name is not null"
}

In [0]:
# STAGING streaming table — ingest silver + enforce quality
#    @dlt.table            -> defines a stored streaming table
#    @dlt.expect_all_or_drop-> rows failing ANY rule are DROPPED
#    (3 actions: expect=warn | _or_drop=drop | _or_fail=stop)

@dlt.table()
@dlt.expect_all_or_drop(my_rules)
def DimProducts_stage():
    return spark.readStream.table("databricks_cata.silver.products_silver ")


In [0]:
#VIEW — intermediate, non-persisted step
#    'Live.' references another dataset in THIS pipeline,
#    which is how DLT auto-builds the dependency graph.

@dlt.view()
def DimProducts_view():
    return spark.readStream.table("Live.DimProducts_stage")

In [0]:
#Creating the empty target dimension table
#apply_changes needs a pre-declared streaming table to write into

dlt.create_streaming_table("DimProducts")

In [0]:
#apply_changes — the SCD engine (one call = full SCD2)
"""
keys           = business key to match on
sequence_by    = recency column (which version is newer)
stored_as_scd_type = 2  -> keep history (1 would overwrite)
except_column_list -> columns ignored when detecting changes
"""

dlt.apply_changes(
    target = "DimProducts",
    source = "DimProducts_view",
    keys = ["product_id"],
    sequence_by = "product_id", #date column should be mentioned
    stored_as_scd_type = 2,
    except_column_list = None 

)
